In [5]:
import os
import pandas as pd
import numpy as np

def extract_axis_features_from_folder(folder_path):
    """
    Iterates through a folder of CSV files.
    Extracts Mean, Std, Min, Max for the first 6 columns (sensor data).
    Returns a DataFrame where every row is a file's 24-feature vector.
    """
    
    features_list = []
    file_names = []
    
    # Define column names for clarity. ADJUST THESE if your CSV headers are different.
    # We assume the order is: Acc_X, Acc_Y, Acc_Z, Gyro_X, Gyro_Y, Gyro_Z 
    # (after Time/Timestamp column, if present).
    col_labels = ['Acc_X', 'Acc_Y', 'Acc_Z', 'Gyro_X', 'Gyro_Y', 'Gyro_Z']
    
    print(f"Starting feature extraction from folder: {folder_path}")
    
    # Loop through every file in the directory
    for file in os.listdir(folder_path):
        if file.endswith(".csv"):
            file_path = os.path.join(folder_path, file)
            
            try:
                # Load the CSV file
                # Phyphox CSVs often have a 'Time (s)' column first.
                df = pd.read_csv(file_path, sep=';', decimal=',')
                
                # Check column count and skip the first (Time) column
                if df.shape[1] >= 7:
                    # Select 6 sensor columns (from index 1 to 6, excluding Time at index 0)
                    data = df.iloc[:, 1:7]
                elif df.shape[1] == 6:
                    # Select all 6 columns if no Time column is present
                    data = df.iloc[:, :6]
                else:
                    print(f"Skipping {file}: Not enough columns found ({df.shape[1]} columns).")
                    continue
                
                if data.empty:
                    print(f"Skipping {file}: Data frame is empty after selection.")
                    continue

                # --- FEATURE CALCULATION ---
                # Calculate the 4 required statistics for the 6 axes
                means = data.mean().values
                stds = data.std().values
                mins = data.min().values
                maxs = data.max().values
                
                # Concatenate them into one long feature vector (4 stats * 6 axes = 24 features)
                # Order: [Mean_X...Mean_Z, Std_X...Std_Z, Min_X...Min_Z, Max_X...Max_Z]
                feature_vector = np.concatenate([means, stds, mins, maxs])
                
                features_list.append(feature_vector)
                file_names.append(file)
                
            except Exception as e:
                print(f"Error processing {file}: {e}")

    # Create the column headers for the final DataFrame
    stats = ['Mean', 'Std', 'Min', 'Max']
    final_headers = [f"{stat}_{col}" for stat in stats for col in col_labels]
    
    # Create a Pandas DataFrame for the results
    result_df = pd.DataFrame(features_list, columns=final_headers, index=file_names)
    
    return result_df

# --- USAGE SECTION ---

# 1. CHANGE THIS PATH to the folder containing your Phyphox CSV files
my_folder_path = './HumanActivity/Jumping'

if os.path.exists(my_folder_path):
    # 2. Run the function
    features_df = extract_axis_features_from_folder(my_folder_path)
    
    # Print the result (First 5 rows)
    print("\nFeatures extracted successfully:")
    print(features_df.head())
    
    # 3. Optional: Save the features to a new CSV file ready for training
    output_file_name = 'extracted_HAR_features_24D.csv'
    features_df.to_csv(output_file_name)
    print(f"\nFeatures saved to {output_file_name}")
else:
    print(f"ERROR: The specified folder path does not exist: {my_folder_path}")

Starting feature extraction from folder: ./HumanActivity/Jumping

Features extracted successfully:
                    Mean_Acc_X  Mean_Acc_Y  Mean_Acc_Z  Mean_Gyro_X  \
Jumping Haakon.csv    -5.89196   -7.634484   -3.685534      0.01003   

                    Mean_Gyro_Y  Mean_Gyro_Z  Std_Acc_X  Std_Acc_Y  Std_Acc_Z  \
Jumping Haakon.csv    -0.118189    -0.067262   9.606667  13.088459  14.188164   

                    Std_Gyro_X  ...  Min_Acc_Z  Min_Gyro_X  Min_Gyro_Y  \
Jumping Haakon.csv    3.795342  ... -78.244904  -18.832823  -16.059113   

                    Min_Gyro_Z  Max_Acc_X  Max_Acc_Y  Max_Acc_Z  Max_Gyro_X  \
Jumping Haakon.csv   -4.424802  15.771205  10.460567  62.945358   10.534524   

                    Max_Gyro_Y  Max_Gyro_Z  
Jumping Haakon.csv   10.399294   10.690447  

[1 rows x 24 columns]

Features saved to extracted_HAR_features_24D.csv
